# Lesson 07 — Python Live Coding Prep

Covers: patterns interviewers test — vectorisation, generators, decorators, ML algorithm implementations from scratch.

**The rule:** write production-quality code, not pseudocode. Name variables well, handle edge cases, state complexity.

## 1. Vectorisation: NumPy vs Python loops

In [ ]:
import numpy as np
import time

def pairwise_distances_loop(X):
    """O(n^2) Python loop — painfully slow."""
    n = len(X)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            D[i, j] = np.sqrt(np.sum((X[i] - X[j])**2))
    return D

def pairwise_distances_numpy(X):
    """Vectorised: (a-b)^2 = a^2 - 2ab + b^2  ->  O(n^2*d) but no Python loops."""
    sq = np.sum(X**2, axis=1, keepdims=True)     # (n, 1)
    return np.sqrt(np.maximum(sq + sq.T - 2 * X @ X.T, 0))  # clip for numerical safety

X = np.random.randn(200, 50)

t0 = time.perf_counter()
D_loop = pairwise_distances_loop(X)
t1 = time.perf_counter()
D_vec  = pairwise_distances_numpy(X)
t2 = time.perf_counter()

print(f"Loop:  {t1-t0:.3f}s")
print(f"NumPy: {t2-t1:.4f}s  ({(t1-t0)/(t2-t1):.0f}x speedup)")
print(f"Max abs diff: {np.max(np.abs(D_loop - D_vec)):.2e}")


## 2. K-Means from Scratch

In [ ]:
import numpy as np

def kmeans(X: np.ndarray, k: int, max_iter: int = 100, tol: float = 1e-4, seed: int = 42):
    """
    Lloyd's algorithm.
    Returns:
        centroids: (k, d)
        labels:    (n,)
    """
    rng = np.random.default_rng(seed)
    # k-means++ init: spread initial centroids
    idx = [rng.integers(len(X))]
    for _ in range(k - 1):
        dists = np.min(np.linalg.norm(X[:, None] - X[idx], axis=2)**2, axis=1)
        probs = dists / dists.sum()
        idx.append(rng.choice(len(X), p=probs))
    centroids = X[idx].copy()

    for _ in range(max_iter):
        # Assignment step
        dists = np.linalg.norm(X[:, None] - centroids[None], axis=2)  # (n, k)
        labels = dists.argmin(axis=1)

        # Update step
        new_centroids = np.array([X[labels == j].mean(axis=0) if (labels == j).any()
                                  else centroids[j] for j in range(k)])

        if np.max(np.abs(new_centroids - centroids)) < tol:
            break
        centroids = new_centroids

    return centroids, labels

# Test
X = np.vstack([np.random.randn(100, 2) + c for c in [(0,0),(5,0),(2.5,4)]])
centroids, labels = kmeans(X, k=3)
print(f"Centroids:\n{centroids.round(2)}")
print(f"Cluster sizes: {np.bincount(labels)}")


## 3. Decorators & Context Managers (Python Fluency)

In [ ]:
import time
import functools
from contextlib import contextmanager

# --- Timing decorator ---
def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

slow_sum(10_000_000)

# --- Retry decorator ---
def retry(max_attempts=3, exceptions=(Exception,)):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if attempt == max_attempts - 1:
                        raise
                    print(f"Attempt {attempt+1} failed: {e}. Retrying...")
        return wrapper
    return decorator

# --- Context manager for temporary numpy print settings ---
@contextmanager
def numpy_print_precision(decimals=2):
    old = np.get_printoptions()
    np.set_printoptions(precision=decimals, suppress=True)
    try:
        yield
    finally:
        np.set_printoptions(**old)

import numpy as np
x = np.array([1.23456789, 9.87654321])
with numpy_print_precision(2):
    print(x)  # [1.23 9.88]
print(x)      # full precision restored


## 4. Generators & Memory-Efficient Data Processing

In [ ]:
import numpy as np
from typing import Generator

def batch_generator(X: np.ndarray, y: np.ndarray,
                    batch_size: int, shuffle: bool = True) -> Generator:
    """Yield mini-batches without loading all data into memory at once."""
    n = len(X)
    indices = np.random.permutation(n) if shuffle else np.arange(n)
    for start in range(0, n, batch_size):
        idx = indices[start:start + batch_size]
        yield X[idx], y[idx]

X = np.random.randn(1000, 10)
y = np.random.randint(0, 3, 1000)

batches = list(batch_generator(X, y, batch_size=32))
print(f"Number of batches: {len(batches)}")
print(f"Last batch size: {len(batches[-1][0])}")  # may be < 32

# Generator for reading large files line by line
def read_large_file(filepath):
    """Memory-efficient file reading."""
    with open(filepath) as f:
        for line in f:
            yield line.strip()

# Usage: for line in read_large_file("huge.csv"): process(line)


## 5. Implementing Linear Regression (Normal Equation + Gradient Descent)

In [ ]:
import numpy as np

class LinearRegression:
    def __init__(self, method="gd", lr=0.01, n_iter=1000, fit_intercept=True):
        self.method = method
        self.lr = lr
        self.n_iter = n_iter
        self.fit_intercept = fit_intercept

    def fit(self, X, y):
        if self.fit_intercept:
            X = np.c_[np.ones(len(X)), X]

        if self.method == "normal":
            # w = (X^T X)^{-1} X^T y — exact solution, O(d^3)
            self.w = np.linalg.lstsq(X, y, rcond=None)[0]

        elif self.method == "gd":
            self.w = np.zeros(X.shape[1])
            n = len(X)
            for _ in range(self.n_iter):
                grad = X.T @ (X @ self.w - y) / n
                self.w -= self.lr * grad
        return self

    def predict(self, X):
        if self.fit_intercept:
            X = np.c_[np.ones(len(X)), X]
        return X @ self.w

# Test
rng = np.random.default_rng(0)
X = rng.standard_normal((200, 3))
true_w = np.array([1.5, -2.0, 0.8])
y = X @ true_w + 0.5 + rng.standard_normal(200) * 0.1

for method in ["normal", "gd"]:
    model = LinearRegression(method=method, lr=0.1, n_iter=500)
    model.fit(X, y)
    preds = model.predict(X)
    mse = np.mean((preds - y)**2)
    print(f"{method:8s}: MSE={mse:.4f}, weights={model.w[1:].round(3)}")


## 6. Common Interview Patterns — Quick Reference

### Time/Space Complexity of ML Operations

| Operation | Time | Space |
|---|---|---|
| Matrix multiply (n×d) × (d×k) | O(ndk) | O(nk) |
| KNN prediction (n train, d dims) | O(nd) per query | O(nd) |
| K-Means one iteration | O(nkd) | O(nk) |
| Attention (seq S, dim D) | O(S²D) | O(S²) |
| Backprop through linear layer | O(batch × in × out) | Same as forward |

### Python Gotchas Interviewers Test

```python
# 1. Mutable default argument
def bad(lst=[]):    # WRONG — same list object reused across calls
    lst.append(1)
    return lst

def good(lst=None):
    if lst is None:
        lst = []
    lst.append(1)
    return lst

# 2. Late binding in closures
fns_bad  = [lambda: i for i in range(3)]     # all return 2
fns_good = [lambda i=i: i for i in range(3)] # 0, 1, 2

# 3. Integer division
10 / 3   # 3.333...  (float)
10 // 3  # 3         (floor div)

# 4. is vs ==
a = [1, 2]
b = [1, 2]
print(a == b)   # True  (value equality)
print(a is b)   # False (identity)
```

## Interview Q&A

**Q: What's the complexity of your k-means implementation?**  
Each iteration: O(n × k × d) for assignment (n points, k centroids, d dims) + O(n × d) for centroid update. Total: O(I × n × k × d) for I iterations. Space: O(n × d) for data + O(k × d) for centroids.

**Q: When would you use the normal equation over gradient descent?**  
Normal equation: exact, O(d³) — fine for d < 10k. Gradient descent: O(n × d) per epoch — better when n is large, d is large, or you want online updates. Normal equation is also numerically unstable if X^TX is near-singular (use `lstsq` with rcond, not `inv`).

**Q: How do you handle imbalanced classes?**  
1. Class weights in loss (`weight` param in `nn.CrossEntropyLoss`). 2. Oversample minority (`SMOTE`). 3. Undersample majority. 4. Change evaluation metric to F1/PR-AUC instead of accuracy. 5. Threshold tuning on validation set.